# FACTOR × Descoeudres et al. (2026.03) 真实数据集与 35 个真实方法交互演示

本 Notebook 直接以 2026 年 3 月工作的两个官方开源仓库为底座：
1. **真实评测与方法库**：`external/spatial-domain-analysis-framework`（含原始 `osmFISH_Codeluppi` 单细胞切片矩阵、跨 8 种技术 94 张真实切片与 930+ 半合成切片的 `scores.csv`，以及 **35 个真实 `.py`/`.R` 方法脚本 + `chimera_run.py` 六大可插拔深度模型**）；
2. **真实单细胞核底座与半合成模拟器**：`external/spatial-domain-simulator`（含小鼠全脑 snRNA-seq `subset_anndata.h5ad.gz` 与官方受控扰动生成器）。

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from factor_demo.canzar_bridge.canzar_data_loader import (
    DEFAULT_FRAMEWORK_DIR,
    inspect_raw_3csv_slice_pure_python,
    load_canzar_real_benchmark_records,
)
from factor_demo.canzar_bridge.canzar_ast_inspector import inspect_all_canzar_methods

## Step 1：`Dataset-Character Agent` 读取真实 `osmFISH_Codeluppi` 原始表达矩阵与 82 张跨技术真实切片元特征

In [ ]:
osm_stats = inspect_raw_3csv_slice_pure_python(DEFAULT_FRAMEWORK_DIR)
print("原始 osmFISH 切片实测统计:", json.dumps(osm_stats, indent=2, ensure_ascii=False))

raw_rows, d_obs_map, d_ref_map = load_canzar_real_benchmark_records(DEFAULT_FRAMEWORK_DIR, exclude_st_stahl=True)
df_slices = pd.DataFrame([
    {
        "sample_uid": s,
        "dataset": s.split("::")[0],
        **{f"obs:{k}": v for k, v in d_obs_map[s].items()},
        **{f"ref:{k}": v for k, v in d_ref_map[s].items()},
    }
    for s in list(d_obs_map.keys())
])
display(df_slices.groupby("dataset").mean(numeric_only=True).round(3))

## Step 2：`Method-Character Agent` 静态解析全部 35 个真实方法脚本与 `chimera_run.py` 模块接口

In [ ]:
method_inspections = inspect_all_canzar_methods(DEFAULT_FRAMEWORK_DIR)
df_methods = pd.DataFrame([
    {
        "method_id": m["method_id"],
        "family": m["family"],
        "script": m["script_file"],
        **m["m_core_tristate"],
        **m["c_aux_tristate"],
        "n_unknown": len(m["unknown_fields"]),
        "chimera_hooks": ", ".join(m["chimera_hooks"]),
        "code_spans": "; ".join(f"{v.file_path}:L{v.start_line}" for v in list(m["evidence_spans"].values())[:3]),
    }
    for m in method_inspections
])
display(df_methods)

## Step 3：读取完整三智能体实测报告（探索集筛选 + `exp8`/`exp2` 2×2 因子对比 + 独立确认集验证）

In [ ]:
import subprocess

subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts" / "run_canzar2026_real_audit.py")], check=True)
report_md = (PROJECT_ROOT / "outputs" / "canzar2026_real_audit" / "canzar2026_real_agent_report.md").read_text(encoding="utf-8")
display(Markdown(report_md))

## Step 4：（在配好 `Chimera` / `scanpy+pytorch` 的服务器上）对真实切片与 `subset_anndata.h5ad.gz` 跑四臂反事实新实验

取消下方注释即可在服务器上直接调用 `GraphST` / `SEDR` / `STAGATE` / `SpaceFlow` 跑 `orig` vs. `refac` vs. `self` ($\mathbf{I}$) vs. `null` ($\mathbf{P}\mathbf{A}\mathbf{P}^\top$) 四臂对照实验：

In [ ]:
# from factor_demo.canzar_bridge.chimera_counterfactual_driver import run_chimera_counterfactual
# base = DEFAULT_FRAMEWORK_DIR / "resources" / "osmFISH_Codeluppi" / "correctedsample"
# for arm in ("orig", "refac", "self", "null"):
#     out = run_chimera_counterfactual(
#         umi_csv=f"{base}_UMImatrix.csv",
#         coords_csv=f"{base}_coords.csv",
#         n_domains=11,
#         preprocess_mod="SEDR",
#         adjacency_mod="SEDR",
#         train_mod="SEDR",
#         clust_mod="kmeans",
#         arm=arm,
#     )
#     print(arm, out["clustering_tsv"])